In [1]:
import torch
from cuda_mig import MIGManager

MIGManager.setup_cuda_devices()
# for i in range(torch.cuda.device_count()):
#     free_mem, total_mem = torch.cuda.mem_get_info(i)
#     print(f"🔹 GPU {i}: {torch.cuda.get_device_name(i)}")
#     print(f"   Свободно: {free_mem / 1e9:.2f} GB / {total_mem / 1e9:.2f} GB")

In [2]:
values_list=['Self-direction','Stimulation','Hedonism','Achievement','Power','Security','Conformity','Tradition','Benevolence','Universalism']

In [3]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.optim as optim
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve
)

In [ ]:
import pandas as pd
#load training data: df_train
#load main expert benchmark (N=1,000): df_eval

In [7]:
def parse_soft_with_expert(row, value):

    # --- aggregated labels ---
    original = row[f"{value}_original"]
    final = row[f"{value}_final"]

    # --- if expert changed the label during expert validation---
    if pd.notna(final) and original != final:
        return float(final)

    # --- otherwise compute real soft from 5 votes ---
    raw_votes = str(row[value]).split(",")
    votes = list(map(float, raw_votes))
    s = sum(votes)

    return s / 5.0

def build_soft_matrix(df, values):
    matrix = []
    for v in values:
        matrix.append(df[v].map(parse_soft))
    return np.vstack(matrix).T

In [8]:
#=====   soft labels =====
for v in values_list:
    df_train[f"{v}_soft"] = df_train.apply(
        lambda row: parse_soft_with_expert(row, v),
        axis=1
    )
y=df_train[[f"{v}_soft" for v in values_list]].to_numpy()
#==============================


In [10]:
class MultiLabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32)
        }

In [11]:
class BERTMultiLabel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask)
        mean_pooled = outputs.last_hidden_state.mean(dim=1)
        logits = self.classifier(self.dropout(mean_pooled))
        return logits

In [12]:
def unfreeze_last_layers(model, n_layers=2):
    for param in model.bert.parameters():
        param.requires_grad = False

    for layer in model.bert.encoder.layer[-n_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    df_train,
    y,
    test_size=0.2,
    random_state=42
)

MODEL_NAME = "FacebookAI/xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

train_dataset = MultiLabelDataset(X_train.text.to_list(), y_train, tokenizer)
val_dataset = MultiLabelDataset(X_val.text.to_list(), y_val, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [14]:
class_freq = np.mean(y_train >= 0.6, axis=0)
class_weights = 1 / np.sqrt(class_freq + 1e-6)
class_weights = class_weights / np.max(class_weights)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)


class FocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=0.3):
        super().__init__()
        self.gamma = gamma
        self.class_weights = class_weights
        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        p_t = torch.exp(-bce)
        focal = ((1 - p_t) ** self.gamma) * bce
        
        if self.class_weights is not None:
            focal = focal * self.class_weights
        
        return focal.mean()

In [17]:
model = BERTMultiLabel(MODEL_NAME, num_labels=len(values_list)).to(device)
unfreeze_last_layers(model, n_layers=2)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
import os
from datetime import datetime

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs,
    device,
    save_dir,
    model_tag="baseline"
):

    os.makedirs(save_dir, exist_ok=True)

    best_pr = -np.inf
    history = []

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)

        for batch in progress_bar:

            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        train_loss = total_loss / len(train_loader)

        # ---- Validation ----
        mean_pr, pr_scores, y_prob, y_true, val_loss = evaluate(model, val_loader, device, criterion)

        print(f"\nEpoch {epoch+1}")
        print(f"Train loss: {train_loss:.4f}")
        print(f"Validation loss: {val_loss:.4f}")
        print(f"Mean PR-AUC: {mean_pr:.4f}")

        history.append({
            "epoch": epoch+1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "mean_pr_auc": mean_pr
        })

        # ---- Save every epoch ----
        epoch_path = os.path.join(
            save_dir,
            f"{model_tag}_epoch_{epoch+1}.pt"
        )

        torch.save({
            "epoch": epoch+1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "mean_pr_auc": mean_pr
        }, epoch_path)

        # ---- Save best model ----
        if mean_pr > best_pr:
            best_pr = mean_pr

            best_path = os.path.join(
                save_dir,
                f"{model_tag}_BEST.pt"
            )

            torch.save({
                "epoch": epoch+1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "mean_pr_auc": mean_pr
            }, best_path)

            print("Saved new BEST model")

    return history


def evaluate(model, loader, device, criterion):
    model.eval()

    all_probs = []
    all_labels = []
    total_val_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_val_loss += loss.item()

            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    y_prob = np.vstack(all_probs)
    y_true = np.vstack(all_labels)

    pr_scores = []
    for i in range(y_true.shape[1]):
        pr = average_precision_score(
            (y_true[:, i] >= 0.6).astype(int),
            y_prob[:, i]
        )
        pr_scores.append(pr)

    mean_pr = np.mean(pr_scores)
    val_loss = total_val_loss / len(loader)

    return mean_pr, pr_scores, y_prob, y_true, val_loss

In [25]:
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

def get_predictions(model, df, tokenizer, device, col_suff, batch_size=32):

    dataset = InferenceDataset(df["text"].tolist(), tokenizer)
    loader = DataLoader(dataset, batch_size=batch_size)

    model.eval()
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()

            all_probs.append(probs)

    probs = np.vstack(all_probs)

    for i, v in enumerate(values_list):
        df[f"{v}_bert_{col_suff}"] = probs[:, i]

    return df

In [ ]:

df_test

In [37]:
df_test = get_predictions(model, df_test, tokenizer, device, "soft_bce")

Predicting: 100%|██████████| 125/125 [04:47<00:00,  2.30s/it]


In [ ]:
def prepare_4000(df):
    for v in values_list:
        df[f"{v}_llm_prob"] = df[f"{v}"].apply(aggregate_llm)
    return df


def evaluate_4000(df, col_suff):

    rows = []

    for v in values_list:

        y_llm_soft = df[f"{v}_llm_prob"]
        y_llm_bin = (y_llm_soft >= 0.6).astype(int)
        y_bert = df[f"{v}_bert_{col_suff}"]

        pr = average_precision_score(y_llm_bin, y_bert)
        gap = np.mean(np.abs(y_llm_soft - y_bert))

        rows.append({
            "Value": v,
            "PR BERT→LLM": pr,
            "|LLM-BERT|": gap
        })

    return pd.DataFrame(rows)

In [9]:
values_list=['Self-direction','Stimulation','Hedonism','Achievement','Power','Security','Conformity','Tradition','Benevolence','Universalism']

evaluate_4000(df_test, "soft_bce")

,Value,PR BERT→LLM,|LLM-BERT|
0,Self-direction,0.752168,0.129002
1,Stimulation,0.633071,0.089873
2,Hedonism,0.725530,0.130621
3,Achievement,0.794033,0.085795
4,Power,0.473297,0.060123
5,Security,0.685718,0.136417
6,Conformity,0.429056,0.047411
7,Tradition,0.755163,0.078377
8,Benevolence,0.930178,0.171524
9,Universalism,0.701827,0.099565
